In [ ]:
import gc
import json
import pathlib

import pandas as pd
import plotly.express as px

from kebab.dataset.t_rex.entity_fragment import EntityFragment

Sample element from the dataset
---

In [ ]:
# load the fragments
dataset_path = pathlib.Path.home() / "OneDrive - Microsoft" / "Benchmark" / "Datasets" / "T-REx" / "Fragments" / "2024-06-27" / "full" / "t_rex_entity_fragments.jsonl"

rows = []
with open(dataset_path, encoding="utf-8") as f:
    for line in f:
        fragment = EntityFragment.from_json(line.strip())
        row = {
            "entity_id": fragment.entity_id,
            "names": sorted(fragment.properties["names"] if "names" in fragment.properties else []),
            "properties": {k: sorted(v) for k, v in sorted(fragment.properties.items()) if k != "names"},
        }
        
        rows.append(row)

df = pd.DataFrame(rows)
gc.collect()

print("Example fragment:")
print(fragment)

Fragment statistics
---

In [ ]:
# add counts
df["names_count"] = df["names"].apply(len)
df["properties_count"] = df["properties"].apply(len)
df["property_values_count"] = df["properties"].apply(lambda x: sum(len(v) for v in x.values()))
df[:1]

In [ ]:
print(f"Number of fragments: {len(df):,d}")
print(f"Average number of names per fragment: {df['names_count'].mean():.2f}")
print(f"Average number of properties per fragment: {df['properties_count'].mean():.2f}")
print(f"Average number of property values per fragment: {df['property_values_count'].mean():.2f}")

In [ ]:
# histogram
take = 10
fig = px.histogram(x=df["properties_count"].clip(upper=take-1), nbins=2 * take, title="Fragments by number of Properties",
                   labels={"x": "number of properties", "y": "number of fragments"})

total_entities = len(df)
groups = sorted(df["properties_count"].unique())[:take]
for i in groups:
    count = (df["properties_count"] == i).sum() if i != groups[-1] else (df["properties_count"] >= i).sum()
    fig.add_annotation(x=i, y=count, text=f"{count / total_entities:.1%}",
                       showarrow=False, yshift=10)

fig.update_layout(xaxis=dict(nticks=take))
fig.update_yaxes(title_text="fragment count")
fig.update_layout(width=1000)
fig.show()

In [ ]:
# histogram of names
take = 10
fig = px.histogram(x=df["names_count"].clip(upper=take-1), nbins=2 * take, title="Fragments by number of Names",
                   labels={"x": "number of names", "y": "number of fragments"})

total_entities = len(df)
groups = sorted(df["names_count"].unique())[:take]
for i in groups:
    count = (df["names_count"] == i).sum() if i != groups[-1] else (df["names_count"] >= i).sum()
    fig.add_annotation(x=i, y=count, text=f"{count / total_entities:.1%}",
                       showarrow=False, yshift=10)

fig.update_layout(xaxis=dict(nticks=take))
fig.update_yaxes(title_text="fragment count")
fig.update_layout(width=1000)
fig.show()

Entity statistics
---

In [ ]:
print(f"Number of entities: {df['entity_id'].nunique():,d}")
gdf = df.groupby("entity_id")
print(f"Average number of fragments per entity: {gdf.size().mean():.2f}")
print(f"Entities with at least 2 fragments: {gdf.size().ge(2).sum():,d}")
print(f"Entities with at least 5 fragments: {gdf.size().ge(5).sum():,d}")

In [ ]:
# histogram
take = 10
fig = px.histogram(x=gdf.size().clip(upper=take), nbins=2 * take, title="Entities by number of Fragments",
                   labels={"x": "number of fragments", "y": "number of entities"})

total_entities = len(gdf)
groups = sorted(gdf.size().unique())[:take]
for i in groups:
    count = (gdf.size() == i).sum() if i != groups[-1] else (gdf.size() >= i).sum()
    fig.add_annotation(x=i, y=count, text=f"{count / total_entities:.1%}",
                       showarrow=False, yshift=10)

fig.update_layout(xaxis=dict(nticks=take))
fig.update_yaxes(title_text="entity count")
fig.update_layout(width=1000)
fig.show()

In [ ]:
# median fragment size in each group
median = gdf["properties_count"].median()
median.describe()

In [ ]:
mean = gdf["properties_count"].mean()
mean.describe()

Random sample
---

In [ ]:
# random sample
df[["entity_id", "names", "properties"]].sample(50)

Most popular names
---

In [ ]:
names = []
for n in df["names"]:
    names.extend(n)

names = pd.Series(names)
names_count = names.value_counts()
names_count.columns = ["name", "count"]
names_count[:20]

Fragments with most names
---

In [ ]:
# top fragments by the number of names
df.sort_values("names_count", ascending=False).head(50)